# This notebook is designed to generate explanations for a given dataset.

In [2]:
from openai import OpenAI

import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import json
import sys
import os

# Add parent directory to path to import utils
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from utils import parse_response, serialize_product

# Load OPENAI_API_KEY from .env file
load_dotenv()

client = OpenAI()

## Prompts

In [21]:
# Set the file to apply the explanations for
train_set_path = "../../data/amazon-google/amazon-google-train.json"
train_set = pd.read_json(train_set_path)

In [11]:
train_set.head()

,id_left,title_left,description_left,price_left,cluster_id_left,id_right,title_right,description_right,price_right,cluster_id_right,label,pair_id
0,abt_730,lg 24 ' lds4821ww semi integrated built in whi...,lg 24 ' lds4821ww semi integrated built in whi...,,411,buy_775,lg ldf6920bb fully integrated dishwasher,,,271,0,abt_730#buy_775
1,abt_670,speck seethru clear hard shell case for macboo...,speck seethru clear hard shell case for macboo...,,698,buy_820,speck products seethru case for apple 13 ' mac...,plastic pink,,991,0,abt_670#buy_820
2,abt_497,denon blu-ray disc dvd/cd player dvd3800bdci,denon blu-ray disc dvd/cd player dvd3800bdci 1...,1999.0,115,buy_239,denon dvd-2930ci dvd player dvd2930ci,"dvd + rw , dvd-rw , cd-rw dvd video , dvd audi...",448.0,461,0,abt_497#buy_239
3,abt_644,panasonic dect 6.0 expandable digital cordless...,panasonic dect 6.0 expandable digital cordless...,,70,buy_299,panasonic kx-tg1032s dual handset digital cord...,1 x phone line ( s ) headset jack silver,61.14,891,0,abt_644#buy_299
4,abt_464,sony silver minidv handycam camcorder dcrhc52,sony silver minidv handycam camcorder dcrhc52 ...,,381,buy_32,sony minidv head cleaner dvm12cld,head cleaner,7.95,802,0,abt_464#buy_32


In [12]:
def generate_structured_explanations(product_1, product_2, label, custom_id):
    label = "MATCH" if label == 1 else "NOT A MATCH"
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4.1",
            "messages": [
                {"role": "user", "content": f"""
                Do the two entity descriptions refer to the same real-world entity?
                Entity 1: {product_1}
                Entity 2: {product_2}

                The correct answer is {label}.

                Please provide an explanation for this answer in a structured format, listing the attributes that you compared for reaching this answer. Each attribute should be accompanied by the attribute values and a score between -1 and 1 that shows the importance of the attribute for the decision. If the attribute influenced the decision towards non-match the importance score should be negative. If the attribute pointed towards a match, the importance score should be positive. Also provide a similarity score for the attribute values. If an attribute only occurs in one item, specify the value of that attribute for the other item as "missing". An example output is the following:

                attribute=brand|||importance=0.05|||values=Logitech###Logitech|||similarity=1.00
                attribute=model|||importance=-0.95|||values=MX G500###MX Master 3S|||similarity=0.20
                attribute=color|||importance=0.00|||values=missing###Graphite|||similarity=0.00
                
                Here is a complete example:
                Do the two product descriptions refer to the same real-world product? Entity 1: 'WD 4TB Black My Passport Portable External Hard Drive - USB 3.0 - WDBYFT0040BBK-WESN'. Entity 2: 'Dysk WD My Passport 1TB USB 3.0 black'.
                "No. 
                attribute=brand|||importance=0.05|||values=Western Digital###Western Digital|||similarity=1.00
                attribute=model|||importance=0.95|||values=My Passport###My Passport|||similarity=1.00
                attribute=storage capacity|||importance=0.9|||values=4TB###1TB|||similarity=0.25
                attribute=color|||importance=0.1|||values=Black###Black|||similarity=1.00
                attribute=USB version|||importance=0.05|||values=USB 3.0###USB 3.0|||similarity=1.00
                
                Do not provide a explanation in a different format. The explanation should be in the format described above. Only provide the answer and explanation dont repeat the question.
                """}
            ],
            "max_tokens": 5_000,
            "temperature": 1
        }
    }



In [17]:
def generate_wadhwa_explanations(product_1, product_2, label, custom_id):
    label = "MATCH" if label == 1 else "NOT A MATCH"
    return {
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-mini",
            "messages": [
                {"role": "user", "content": f"""
                <s>[INST] Given the following two examples, provide an explanation for the third example for why the two entities do or do not match. [\INST]

                Entity A: [NAME] samsung dlp tv stand in black tr72bx [DESCRIPTION] samsung dlp tv stand in black tr72bx designed to fit samsung hlt7288, hlt7288, hl72a650, and hl67a650 television sets tempered 6mm tinted glass shelves wide audio storage shelves to accommodate 4 or more components wire management system easy to assemble high gloss black finish [PRICE] 369.0
                Entity B: [NAME] samsung tr72b tv stand [DESCRIPTION] glass black [PRICE] 232.14
                Label: MATCH
                Explanation: Both entities refer to samsung TV stand in black and therefore have substantially similar specifications, therefore they’re a match. </s>

                Entity A: [NAME] canon high capacity color ink cartridge color ink cl51 [DESCRIPTION] canon high capacity color ink cartridge cl51 compatible with pixma ip6210d, ip6220d, mp150, mp170 and mp450 printers [PRICE] 35.0
                Entity B: [NAME] canon pg-40 twin pack black ink cartridge 0615b013 [DESCRIPTION] black [PRICE]
                Label: NOT A MATCH
                Explanation: Entity A refers to color ink cartridge while Entity B is a black ink cartridge, therefore they are not a match. </s>

                Entity A: [NAME] {product_1.get("name")} [DESCRIPTION] {product_1.get("description")} [PRICE] {product_1.get("price")}
                Entity B: [NAME] {product_2.get("name")} [DESCRIPTION] {product_2.get("description")} [PRICE] {product_2.get("price")}
                Label: {label}
                Explanation:
                """}
            ],
            "max_tokens": 128,
            "temperature": 0,
            "top_p": 0.95,
        }
    }

In [14]:
# Function to extract the entity strings
def extract_entities(text):
    entity_1 = text.split("Entity 1: '")[1].split("'")[0]
    entity_2 = text.split("Entity 2: '")[1].split("'")[0]
    return entity_1, entity_2

In [13]:
train_set.head()

,id_left,title_left,description_left,price_left,cluster_id_left,id_right,title_right,description_right,price_right,cluster_id_right,label,pair_id
0,abt_730,lg 24 ' lds4821ww semi integrated built in whi...,lg 24 ' lds4821ww semi integrated built in whi...,,411,buy_775,lg ldf6920bb fully integrated dishwasher,,,271,0,abt_730#buy_775
1,abt_670,speck seethru clear hard shell case for macboo...,speck seethru clear hard shell case for macboo...,,698,buy_820,speck products seethru case for apple 13 ' mac...,plastic pink,,991,0,abt_670#buy_820
2,abt_497,denon blu-ray disc dvd/cd player dvd3800bdci,denon blu-ray disc dvd/cd player dvd3800bdci 1...,1999.0,115,buy_239,denon dvd-2930ci dvd player dvd2930ci,"dvd + rw , dvd-rw , cd-rw dvd video , dvd audi...",448.0,461,0,abt_497#buy_239
3,abt_644,panasonic dect 6.0 expandable digital cordless...,panasonic dect 6.0 expandable digital cordless...,,70,buy_299,panasonic kx-tg1032s dual handset digital cord...,1 x phone line ( s ) headset jack silver,61.14,891,0,abt_644#buy_299
4,abt_464,sony silver minidv handycam camcorder dcrhc52,sony silver minidv handycam camcorder dcrhc52 ...,,381,buy_32,sony minidv head cleaner dvm12cld,head cleaner,7.95,802,0,abt_464#buy_32


In [18]:
# Create the JSONL file with all requests
requests = []
for index, row in tqdm(train_set.iterrows(), total=train_set.shape[0]):
    product_1 = serialize_product(row, "left")
    product_2 = serialize_product(row, "right")
    label = row["label"]
    custom_id = row["pair_id"]
    prompt = generate_structured_explanations(product_1, product_2, label, custom_id=custom_id)
    requests.append(prompt)


batch_file_path = "explanation.jsonl"
with open(batch_file_path, "w") as f:
    for request in requests:
        f.write(json.dumps(request) + "\n")
        

batch_input_file = client.files.create(
    file=open(batch_file_path, "rb"),
    purpose="batch"
)

batch_input_file_id = batch_input_file.id

batch = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"description": "Generate structured explanations for the abt-buy dataset with all attributes"}
)   

  0%|          | 0/7659 [00:00<?, ?it/s]

### Retrieve the batch job from OpenAI
We need the batch ID to check the status and download the results of our explanation generation job


In [ ]:
if 'batch' in locals():
    batch_id = batch.id
else:
    batch_id = "batch_684ffafd9ce48190ac359198134f96a3"

batch = client.batches.retrieve(batch_id)

# Check the status of the batch job
if batch.status == "completed":
    # download the results
    batch_output_file = client.files.retrieve(batch.output_file_id)
    # Get the content of the file
    content = client.files.content(batch.output_file_id)
    # Save the content to a file
    with open(f"explanation_output{batch.id}.jsonl", "wb") as f:
        f.write(content.read())
else:
    print("Batch job not completed. Current status:", batch.status)

### Merging the explanations with the existing training dataset.


In [40]:
batch_id = "batch_684fe6f71abc81909c829e940569e2e6"

In [42]:
generated_explanations = pd.read_json(f"explanation_output{batch_id}.jsonl", lines=True)

generated_explanations_parsed = generated_explanations["response"].apply(parse_response)    
generated_explanations = pd.concat([generated_explanations, generated_explanations_parsed], axis=1)

for index, row in generated_explanations.iterrows():
    explanation = row["content"]
    # find the index with the correct pair id
    index = train_set[train_set["pair_id"] == row["custom_id"]].index[0]
    train_set.at[index, "explanation"] = explanation
    
train_set.to_pickle(train_set_path.replace(".json.gz", "_with_explanation_all_fields_4_1.pkl.gz"), compression="gzip")
train_set.head()


,id_left,title_left,category_left,brand_left,modelno_left,price_left,cluster_id_left,id_right,title_right,category_right,brand_right,modelno_right,price_right,cluster_id_right,label,pair_id,explanation
0,walmart_581,elite screens cinewhite cinema235 series fixed...,electronics - general,elite screens,r85wh1-wide,409.00,847,amazon_17384,cinegray ezframe series fixed frame screen - 1...,projection screens,elite,,879.0,847,0,walmart_581#amazon_17384,No. \nattribute=brand|||importance=0.1|||valu...
1,walmart_1502,san diego padres iphone 4 case silicone cover,electronics - general,tribeca,fva3959,24.99,847,amazon_3825,georgia bulldogs iphone 4 case silicone cover,computers accessories,tribeca,,17.99,847,0,walmart_1502#amazon_3825,No. \nattribute=brand|||importance=0.05|||val...
2,walmart_337,innovera d3010 black compatible high-yield ton...,stationery & office machinery,innovera,d3010,68.35,667,amazon_20538,premium compatible hp 11x toner cartridge hp q...,inkjet printer ink,compatible,hp-q6511x,28.92,847,0,walmart_337#amazon_20538,No. \nattribute=brand|||importance=0.3|||valu...
3,walmart_53,da-lite da-plex base rear projection screen - ...,electronics - general,da-lite,27528,2608.99,595,amazon_13427,da-lite 27514 da-plex unframed rear projection...,projection screens,da-lite,,,847,0,walmart_53#amazon_13427,No. \nattribute=brand|||importance=0.1|||valu...
4,walmart_1560,pc treasures wireless optical mouse 2.4 ghz pu...,mice,pc treasures,07227,17.82,443,amazon_3823,inland pro 2.4 ghz wireless optical mouse,mice,inland,07441,12.98,847,0,walmart_1560#amazon_3823,No. \nattribute=brand|||importance=-0.95|||va...


In [41]:
train_set_path = "../../data/walmart-amazon/walmart-amazon-train.json.gz"
train_set = pd.read_json(train_set_path, compression="gzip")

